# Building a Data Mining Pipeline for Hacker News

### Data Collection, Processing, Analysis and Recommendation System

**Author:** Prantika Biswas

**Project Type:** Data Mining & Exploratory Data Analysis

**Tools & Technologies:** Python, Requests, Pandas, Jupyter Notebook, Hacker News API

**Date:** June 2026


# Introduction

Social media and online discussion platforms generate large volumes of user-created content every day. Mining and analyzing this data can help uncover trends, popular topics, user engagement patterns, and community interests. While the original project focused on Twitter data mining, this project adapts the same data mining concepts using the Hacker News public API, providing a practical alternative for collecting and analyzing real-world social discussion data.

The objective of this project is to build a complete data mining pipeline using Python. The project retrieves live data from Hacker News, processes it into a structured dataset, performs exploratory data analysis, and extracts meaningful insights regarding story popularity, author activity, engagement levels, content categories, keyword trends, and temporal patterns.

The project demonstrates key data mining concepts including data collection through APIs, data cleaning, feature engineering, statistical analysis, trend identification, text analysis, recommendation techniques, and reporting. The entire workflow is implemented using Python, Requests, Pandas, and Jupyter Notebook, providing a reproducible framework for analyzing social and community-driven datasets.


In [1]:
#IMPORTS

import requests
import pandas as pd
import re

from collections import Counter
from datetime import datetime

In [2]:
# DATA COLLECTION FUNCTIONS

def get_top_story_ids():

    url = "https://hacker-news.firebaseio.com/v0/topstories.json"

    return requests.get(url).json()


def get_story(story_id):

    url = f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json"

    return requests.get(
        url,
        timeout=30
    ).json()


def build_dataset(limit=100):

    story_ids = get_top_story_ids()

    stories = []

    for story_id in story_ids[:limit]:

        story = get_story(story_id)

        stories.append({
            "title": story.get("title"),
            "author": story.get("by"),
            "score": story.get("score"),
            "comments": story.get("descendants"),
            "type": story.get("type"),
            "time": story.get("time")
        })

    return pd.DataFrame(stories)

In [3]:
# DATA PROCESSING FUNCTIONS

def categorize_story(title):

    title = str(title).lower()

    if any(word in title for word in [
        "ai",
        "gpt",
        "claude",
        "llm",
        "anthropic",
        "openai"
    ]):
        return "AI"

    elif any(word in title for word in [
        "security",
        "cyber",
        "hack",
        "malware",
        "vulnerability"
    ]):
        return "Cybersecurity"

    elif any(word in title for word in [
        "python",
        "javascript",
        "rust",
        "java",
        "programming",
        "code"
    ]):
        return "Programming"

    elif any(word in title for word in [
        "startup",
        "founder",
        "business"
    ]):
        return "Business"

    return "Other"


def classify_story(score):

    if score >= 1000:
        return "Viral"

    elif score >= 500:
        return "Popular"

    elif score >= 100:
        return "Average"

    else:
        return "Low"


def process_dataset(df):

    df["datetime"] = pd.to_datetime(
        df["time"],
        unit="s"
    )

    df["hour"] = df["datetime"].dt.hour

    df["day"] = df["datetime"].dt.day_name()

    df["popularity"] = df["score"].apply(
        classify_story
    )

    df["category"] = df["title"].fillna("").apply(
        categorize_story
    )

    return df

In [4]:
# Analysis Functions

def dataset_info(df):

    print("Dataset Shape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns)


def basic_stats(df):

    print("\nAverage Score:")
    print(df["score"].mean())

    print("\nAverage Comments:")
    print(df["comments"].mean())

    print("\nHighest Score:")
    print(df["score"].max())

    print("\nHighest Comments:")
    print(df["comments"].max())


def highest_scoring_story(df):

    return (
        df.sort_values(
            by="score",
            ascending=False
        )[["title", "score"]]
        .head(1)
    )


def most_discussed_story(df):

    return (
        df.sort_values(
            by="comments",
            ascending=False
        )[["title", "comments"]]
        .head(1)
    )


def top_stories_by_score(df):

    return (
        df.sort_values(
            by="score",
            ascending=False
        )[["title", "score"]]
        .head(5)
    )


def top_stories_by_comments(df):

    return (
        df.sort_values(
            by="comments",
            ascending=False
        )[["title", "comments"]]
        .head(5)
    )

def search_stories(df, keyword):

    return df[
        df["title"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ][
        ["title", "author", "score", "comments"]
    ]

def recommend_top_stories(
    df,
    min_score=500,
    min_comments=100
):

    recommendations = df[
        (df["score"] >= min_score)
        &
        (df["comments"] >= min_comments)
    ]

    return recommendations.sort_values(
        by="score",
        ascending=False
    )

def category_performance(df):

    return (
        df.groupby("category")
          [["score", "comments"]]
          .mean()
          .sort_values(
              by="score",
              ascending=False
          )
    )

def keyword_impact(
    df,
    keyword
):

    matching = df[
        df["title"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    return {
        "stories": len(matching),
        "avg_score": matching["score"].mean(),
        "avg_comments": matching["comments"].mean()
    }

In [5]:
# TEXT MINING FUNCTIONS

def most_common_words(df):

    all_titles = " ".join(
        df["title"].dropna()
    )

    words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        all_titles.lower()
    )

    stop_words = {
        "this", "that", "with", "from",
        "have", "will", "your", "about",
        "into", "they", "their", "what",
        "when", "where", "which", "would",
        "could", "there", "been", "more",
        "than", "after", "before", "under"
    }

    filtered_words = [
        word
        for word in words
        if word not in stop_words
    ]

    word_counts = Counter(filtered_words)

    return word_counts.most_common(20)

def keyword_frequency(df):

    all_titles = " ".join(
        df["title"].dropna()
    )

    words = re.findall(
        r"\b[a-zA-Z]{4,}\b",
        all_titles.lower()
    )

    counts = Counter(words)

    return counts.most_common(50)

In [6]:
# ADVANCED ANALYSIS FUNCTIONS

def correlation_analysis(df):

    return df["score"].corr(
        df["comments"]
    )


def top_authors(df):

    return (
        df["author"]
        .value_counts()
        .head(10)
    )

def author_scores(df, n=10):

    return (
        df.groupby("author")["score"]
          .mean()
          .sort_values(ascending=False)
          .head(n)
    )


def author_comments(df):

    return (
        df.groupby("author")["comments"]
          .mean()
          .sort_values(
              ascending=False
          )
          .head(10)
    )


def stories_by_hour(df):

    return (
        df["hour"]
        .value_counts()
        .sort_index()
    )


def stories_by_day(df):

    return (
        df["day"]
        .value_counts()
    )


def average_score_by_hour(df):

    return (
        df.groupby("hour")["score"]
          .mean()
          .sort_values(
              ascending=False
          )
    )

def category_scores(df):

    return (
        df.groupby("category")["score"]
          .mean()
          .sort_values(
              ascending=False
          )
    )

def category_comments(df):

    return (
        df.groupby("category")["comments"]
          .mean()
          .sort_values(
              ascending=False
          )
    )

def technology_mentions(df):

    technologies = [
        "python",
        "rust",
        "javascript",
        "java",
        "go",
        "linux",
        "ai",
        "openai",
        "claude"
    ]

    counts = {}

    for tech in technologies:

        count = df["title"].str.lower().str.contains(
            tech,
            na=False
        ).sum()

        counts[tech] = count

    return pd.Series(counts).sort_values(
        ascending=False
    )

def author_performance(df):

    return (
        df.groupby("author")
          ["score"]
          .mean()
          .sort_values(
              ascending=False
          )
    )

def category_counts(df):

    return (
        df["category"]
        .value_counts()
    )

def ai_story_analysis(df):

    ai_stories = df[
        df["title"].str.contains(
            "ai|gpt|claude|anthropic|openai",
            case=False,
            na=False,
            regex=True
        )
    ]

    return {
        "count": len(ai_stories),
        "avg_score": ai_stories["score"].mean(),
        "avg_comments": ai_stories["comments"].mean()
    }

def dataset_report(df):

    report = {
        "total_stories": len(df),
        "avg_score": df["score"].mean(),
        "avg_comments": df["comments"].mean(),
        "max_score": df["score"].max(),
        "max_comments": df["comments"].max(),
        "unique_authors": df["author"].nunique(),
        "categories": df["category"].nunique()
    }

    return report

In [7]:
# AI TOPIC DETECTION

def find_ai_stories(df):

    ai_keywords = [
        "ai",
        "claude",
        "gpt",
        "llm",
        "anthropic",
        "openai"
    ]

    return df[
        df["title"].fillna("").str.lower()
        .apply(
            lambda title:
            any(
                keyword in title
                for keyword in ai_keywords
            )
        )
    ]

In [8]:
# BUILD DATASET

df = build_dataset(20)

df = process_dataset(df)

df.to_csv(
    "top_stories.csv",
    index=False
)

print("Dataset Saved Successfully")

dataset_info(df)

basic_stats(df)

Dataset Saved Successfully
Dataset Shape:
(20, 11)

Columns:
Index(['title', 'author', 'score', 'comments', 'type', 'time', 'datetime',
       'hour', 'day', 'popularity', 'category'],
      dtype='str')

Average Score:
372.65

Average Comments:
173.3

Highest Score:
1269

Highest Comments:
394


In [9]:
print("\nHighest Scoring Story")
display(highest_scoring_story(df))

print("\nMost Discussed Story")
display(most_discussed_story(df))

print("\nTop Stories By Score")
display(top_stories_by_score(df))

print("\nTop Stories By Comments")
display(top_stories_by_comments(df))


Highest Scoring Story


,title,score
4,Show HN: Homebrew 6.0.0,1269



Most Discussed Story


,title,comments
9,Claude Fable is relentlessly proactive,394



Top Stories By Score


,title,score
4,Show HN: Homebrew 6.0.0,1269
2,"If you are asking for human attention, demonst...",854
0,AI agent bankrupted their operator while tryin...,703
11,MiMo Code is now released and open-source,506
3,Nobody ever gets credit for fixing problems th...,499



Top Stories By Comments


,title,comments
9,Claude Fable is relentlessly proactive,394
10,Anthropic apologizes for invisible Claude Fabl...,393
4,Show HN: Homebrew 6.0.0,307
2,"If you are asking for human attention, demonst...",287
0,AI agent bankrupted their operator while tryin...,280


In [10]:
print("\nTop Authors")
display(top_authors(df))

print("\nAuthor Scores")
display(author_scores(df))

print("\nAuthor Comments")
display(author_comments(df))


Top Authors


author
xiaoyu2006       1
soheilpro        1
jjfoooo4         1
sam_bristow      1
mikemcquaid      1
keyle            1
msephton         1
matthewbarras    1
sneela           1
lumpa            1
Name: count, dtype: int64


Author Scores


author
mikemcquaid      1269.0
jjfoooo4          854.0
xiaoyu2006        703.0
apeters           506.0
sam_bristow       499.0
lumpa             491.0
hmokiguess        443.0
rarisma           437.0
matthewbarras     416.0
RyeCombinator     398.0
Name: score, dtype: float64


Author Comments


author
lumpa            394.0
rarisma          393.0
mikemcquaid      307.0
jjfoooo4         287.0
xiaoyu2006       280.0
apeters          279.0
RyeCombinator    277.0
matthewbarras    225.0
sam_bristow      167.0
bugvader         163.0
Name: comments, dtype: float64

In [11]:
print("\nStory Types")
display(df["type"].value_counts())

print("\nCorrelation")
print(correlation_analysis(df))


Story Types


type
story    20
Name: count, dtype: int64


Correlation
0.7419730965405855


In [12]:
print("\nMost Common Words")
display(most_common_words(df))

print("\nKeyword Frequency")
display(keyword_frequency(df))

print("\nTechnology Mentions")
display(technology_mentions(df))


Most Common Words


[('fable', 4),
 ('claude', 3),
 ('human', 2),
 ('show', 2),
 ('code', 2),
 ('agent', 1),
 ('bankrupted', 1),
 ('operator', 1),
 ('while', 1),
 ('trying', 1),
 ('scan', 1),
 ('future', 1),
 ('email', 1),
 ('asking', 1),
 ('attention', 1),
 ('demonstrate', 1),
 ('effort', 1),
 ('nobody', 1),
 ('ever', 1),
 ('gets', 1)]


Keyword Frequency


[('fable', 4),
 ('claude', 3),
 ('human', 2),
 ('show', 2),
 ('than', 2),
 ('code', 2),
 ('from', 2),
 ('agent', 1),
 ('bankrupted', 1),
 ('their', 1),
 ('operator', 1),
 ('while', 1),
 ('trying', 1),
 ('scan', 1),
 ('future', 1),
 ('email', 1),
 ('asking', 1),
 ('attention', 1),
 ('demonstrate', 1),
 ('effort', 1),
 ('nobody', 1),
 ('ever', 1),
 ('gets', 1),
 ('credit', 1),
 ('fixing', 1),
 ('problems', 1),
 ('that', 1),
 ('never', 1),
 ('happened', 1),
 ('homebrew', 1),
 ('packages', 1),
 ('compromised', 1),
 ('with', 1),
 ('infostealer', 1),
 ('rootkit', 1),
 ('made', 1),
 ('video', 1),
 ('game', 1),
 ('prince', 1),
 ('persia', 1),
 ('fablepool', 1),
 ('pool', 1),
 ('money', 1),
 ('behind', 1),
 ('prompt', 1),
 ('builds', 1),
 ('public', 1),
 ('vinyl', 1),
 ('succumbs', 1),
 ('loudness', 1)]


Technology Mentions


ai            4
claude        3
go            1
linux         1
python        0
java          0
javascript    0
rust          0
openai        0
dtype: int64

In [13]:
print("\nStories By Hour")
display(stories_by_hour(df))

print("\nStories By Day")
display(stories_by_day(df))

print("\nAverage Score By Hour")
display(average_score_by_hour(df))


Stories By Hour


hour
0     2
1     1
4     1
5     1
7     1
8     1
10    1
12    2
13    1
14    1
15    2
16    2
18    1
21    1
22    1
23    1
Name: count, dtype: int64


Stories By Day


day
Thursday    9
Friday      7
Tuesday     2
Saturday    1
Monday      1
Name: count, dtype: int64


Average Score By Hour


hour
13    1269.0
23     854.0
4      703.0
14     506.0
1      491.0
12     417.5
21     416.0
15     384.0
0      299.0
16     295.0
22     147.0
18      84.0
7       61.0
5       52.0
10      42.0
8       37.0
Name: score, dtype: float64

In [14]:
print("\nAI Stories")
display(
    find_ai_stories(df)[
        ["title", "score"]
    ]
)

print("\nAI KEYWORD IMPACT\n")
print(keyword_impact(df, "AI"))

print("\nAI STORY ANALYSIS\n")
print(ai_story_analysis(df))


AI Stories


,title,score
0,AI agent bankrupted their operator while tryin...,703
1,The Future of Email,42
9,Claude Fable is relentlessly proactive,491
10,Anthropic apologizes for invisible Claude Fabl...,437
16,Claude Fable 5: mid-tier results on coding tasks,335
17,Ear Training Practice,255
18,Making a vintage LLM from scratch,37



AI KEYWORD IMPACT

{'stories': 4, 'avg_score': np.float64(359.25), 'avg_comments': np.float64(203.5)}

AI STORY ANALYSIS

{'count': 6, 'avg_score': np.float64(377.1666666666667), 'avg_comments': np.float64(228.5)}


In [15]:
print("\nCategory Counts")
display(category_counts(df))

print("\nCategory Scores")
display(category_scores(df))

print("\nCategory Comments")
display(category_comments(df))

print("\nCATEGORY PERFORMANCE\n")
print(category_performance(df))


Category Counts


category
Other          11
AI              7
Programming     2
Name: count, dtype: int64


Category Scores


category
Programming    452.000000
Other          386.272727
AI             328.571429
Name: score, dtype: float64


Category Comments


category
Programming    278.000000
AI             196.857143
Other          139.272727
Name: comments, dtype: float64


CATEGORY PERFORMANCE

                  score    comments
category                           
Programming  452.000000  278.000000
Other        386.272727  139.272727
AI           328.571429  196.857143


In [16]:
print("\nTOP RECOMMENDED STORIES\n")

display(
    recommend_top_stories(df)
    [["title", "score", "comments"]]
    .head(10)
)

print("\nPYTHON STORIES\n")

display(
    search_stories(df, "Python")
)


TOP RECOMMENDED STORIES



,title,score,comments
4,Show HN: Homebrew 6.0.0,1269,307
2,"If you are asking for human attention, demonst...",854,287
0,AI agent bankrupted their operator while tryin...,703,280
11,MiMo Code is now released and open-source,506,279



PYTHON STORIES



,title,author,score,comments


In [17]:
print("\nDATASET REPORT\n")
print(dataset_report(df))


DATASET REPORT

{'total_stories': 20, 'avg_score': np.float64(372.65), 'avg_comments': np.float64(173.3), 'max_score': np.int64(1269), 'max_comments': np.int64(394), 'unique_authors': 20, 'categories': 3}


# Conclusion

This project successfully implemented a complete social data mining workflow using the Hacker News public API. Data from the platform was collected, transformed into a structured dataset, and analyzed using various statistical and exploratory techniques. Through the analysis, valuable insights were obtained regarding story popularity, user engagement, author contributions, content categories, keyword frequency, and posting patterns.

The project demonstrated how raw online discussion data can be converted into meaningful information through data preprocessing, feature engineering, aggregation, correlation analysis, and trend detection. Additional analyses such as category-based performance evaluation, keyword impact assessment, recommendation generation, and temporal analysis further enhanced the depth of the study.

Overall, the project provided practical experience in API-based data collection, data wrangling, exploratory data analysis, and social media mining techniques. The developed notebook serves as a reusable foundation for future projects involving larger datasets, advanced visualizations, machine learning models, sentiment analysis, or real-time trend monitoring systems.
